### 1️⃣ Load Required Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, r2_score

pd.set_option('display.max_columns', None)

### 2️⃣ Load Dataset

In [2]:
df = pd.read_csv("cleaned_train_data.csv")
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize
0,0,3,1,22.0,1,0,7.2500,2,2
1,1,1,0,38.0,1,0,65.6344,0,2
2,1,3,0,26.0,0,0,7.9250,2,1
3,1,1,0,35.0,1,0,53.1000,2,2
4,0,3,1,35.0,0,0,8.0500,2,1


### 3️⃣ Separate Features & Target

#### 🔹 Classification Target

In [3]:
X = df.drop('Survived', axis=1)
y_classification = df['Survived']

#### 🔹 Regression Target

In [4]:
y_regression = df['Fare']

### 4️⃣ Identify Feature Types

In [5]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['category', 'object']).columns

numeric_features, categorical_features

(Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
        'FamilySize'],
       dtype='object'),
 Index([], dtype='object'))

### 5️⃣ Preprocessing Pipelines

#### 🔹 Numerical Pipeline

In [6]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#### 🔹 Categorical Pipeline

In [7]:
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

### 6️⃣ Combine Using ColumnTransformer

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

### 7️⃣ Classification Pipeline (Logistic Regression)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_classification, test_size=0.2, random_state=42, stratify=y_classification
)

clf_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

clf_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
       'FamilySize'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index([], dtype='object'))])),
                ('model', LogisticRegression(max_iter=1000))])

### 8️⃣ Train & Evaluate Classification Pipeline

In [10]:
clf_pipeline.fit(X_train, y_train)

y_pred = clf_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7932960893854749
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       110
           1       0.77      0.67      0.71        69

    accuracy                           0.79       179
   macro avg       0.79      0.77      0.78       179
weighted avg       0.79      0.79      0.79       179



### 9️⃣ Classification Pipeline – Random Forest

In [11]:
rf_clf_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=42
    ))
])

rf_clf_pipeline.fit(X_train, y_train)
rf_pred = rf_clf_pipeline.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.8044692737430168


### 🔟 Regression Pipeline (Predict Fare)

In [12]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_regression, test_size=0.2, random_state=42
)

reg_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        random_state=42
    ))
])

reg_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
       'FamilySize'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index([], dtype='object'))])),
                ('model',
                 RandomForestRegressor(max_depth=8, n_estimators=200,
                                       random_state=42))])

### 1️⃣1️⃣ Train & Evaluate Regression Pipeline

In [13]:
reg_pipeline.fit(X_train_r, y_train_r)

y_pred_r = reg_pipeline.predict(X_test_r)

print("R2 Score:", r2_score(y_test_r, y_pred_r))

R2 Score: 0.999932971081115


### 1️⃣2️⃣ Pipeline + GridSearchCV

#### 🔹 Hyperparameter Tuning

In [14]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [4, 6, 8]
}

grid_search = GridSearchCV(
    rf_clf_pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
       'FamilySize'],
      dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         Index([], dtype='object'))])),
                                       ('model',
                                        RandomForestClassifier(max_depth=6,
                                                               n_estimators=200,
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [4, 6, 8],
                         'model__n_estimators': [100, 200]},
             scoring='accuracy')

#### 🔹 Best Model

In [15]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

Best Parameters: {'model__max_depth': 6, 'model__n_estimators': 200}
Best CV Accuracy: 0.823116320299419


### 1️⃣3️⃣ Use Best Pipeline for Prediction

In [16]:
best_pipeline = grid_search.best_estimator_

best_pipeline.predict(X_test[:5])

array([0, 0, 0, 0, 1], dtype=int64)

### 1️⃣4️⃣ Save Pipeline (Production Ready)

In [17]:
import joblib

joblib.dump(best_pipeline, "titanic_classification_pipeline.pkl")

['titanic_classification_pipeline.pkl']

### 1️⃣5️⃣ Load Pipeline & Predict (Simulation)

In [18]:
loaded_pipeline = joblib.load("titanic_classification_pipeline.pkl")

loaded_pipeline.predict(X_test[:5])

array([0, 0, 0, 0, 1], dtype=int64)